# 1.2a Transform data and engineer features

**Story:** The SafeDrive data is in the lake. Now you turn raw claims and policies into the numeric
features the fraud model will learn from, first by hand in pandas, then with the AWS tools you would
use at scale. You also clean up the live telematics stream with a Lambda function.

**Task statement 1.2:** Perform data transformation, feature engineering, and pre-processing (skills 1.2.1 to 1.2.4;
the GenAI skills 1.2.5 to 1.2.9 are in notebook 1.2b).

| Skill | Topic | Status |
|---|---|---|
| 1.2.1 | Transform with AWS tools: Glue job, DataBrew recipe, SageMaker Processing (EMR, Data Wrangler as notes) | ✅ |
| 1.2.2 | Create and manage features with SageMaker Feature Store | ✅ |
| 1.2.3 | Transform streaming data with Lambda (Spark streaming as notes) | ✅ |
| 1.2.4 | Feature engineering: scaling, standardization, normalization, binning, log transform, splitting, encoding | ✅ |

💰 Cost of a full run: well under USD 1 (three small jobs of a few minutes each).

## Key points for the exam

- **Glue ETL** = serverless Spark jobs (code). **DataBrew** = no-code recipes with 250+ built-in transforms and data profiling.
  **Data Wrangler** (SageMaker Canvas) = visual prep for ML that exports to Pipelines / Feature Store.
  **EMR** = full control of Spark / Hadoop clusters, biggest and most custom workloads.
  **SageMaker Processing** = run your own script (scikit-learn, pandas, Spark) on managed instances.
- **Standardization** (z-score) → mean 0, std 1. **Min-max scaling (normalization)** → range 0 to 1.
  **Robust scaling** uses median and IQR (good with outliers). Tree models (XGBoost) do **not** need scaling;
  linear models, k-NN, k-means, PCA and neural networks do.
- **Log transform** fixes right-skewed values (amounts, income). **Binning** turns a number into categories.
- **One-hot encoding** for categories without order; **ordinal / label encoding** when order matters; watch out for high cardinality.
- **Fit transforms on the training set only**, then apply to validation / test (avoids data leakage).
- **Firehose + Lambda** = simple per-record stream transform. **Flink / Spark Structured Streaming** = windows, joins, state.

| Situation | Choose |
|---|---|
| Analyst cleans data without code | Glue DataBrew |
| Scheduled Spark ETL, catalog integration, serverless | AWS Glue job |
| Existing Spark/Hive code, custom libraries, huge clusters | Amazon EMR |
| Pre-processing step inside a SageMaker pipeline | SageMaker Processing |
| Explore, visualise and export features for ML quickly | SageMaker Data Wrangler |

## Setup

In [ ]:
import sys; sys.path.append("..")
from common import utils as U, catchup as C
print(U.REGION, U.bucket())

Catch-up: the raw files must be in S3. (The feature group from notebook 1.1 is created in section 1.2.2 if it is missing.)

In [ ]:
RAW = C.ensure_raw_data()
print(U.s3_uri(RAW))

Common imports and names.

In [ ]:
import io, json, os, time
import numpy as np
import pandas as pd
from botocore.exceptions import ClientError
from common import features as F

BUCKET, ROLE = U.bucket(), U.role_arn()
TAG_MAP = {t["Key"]: t["Value"] for t in U.TAGS}
PROCESSED = C.PROCESSED            # "safedrive/processed"
os.makedirs("scripts", exist_ok=True)

def exists(fn, **kwargs):
    """True if a describe/get call works, False if the resource is not there."""
    try:
        fn(**kwargs)
        return True
    except ClientError:
        return False

print(ROLE)

## 1.2.4 Feature engineering step by step

We build the fraud-model features by hand. The finished recipe lives in `common/features.py`
(`F.build_features`); at the end we check that our step-by-step result is identical.

| Technique | What it does | Our example |
|---|---|---|
| Log transform | Compress a long right tail | `claim_amount_log` |
| Ratio / interaction | Combine columns into a stronger signal | `claim_to_value_ratio` |
| Feature splitting | One column → several simpler ones | `claim_date` → year, month, weekday |
| Binning | Number → category / flag | `young_driver`, `night_incident` |
| Encoding | Category → numbers | one-hot `claim_type_*` |
| Scaling | Put numeric columns on a similar range | standardization, min-max |

### Load and join
`F.clean_claims` fixes basic quality problems (explained in notebook 1.3). Then every claim gets its policy columns.

In [ ]:
claims = U.read_csv(f"{RAW}/claims.csv")
policies = U.read_csv(f"{RAW}/policies.csv")
df = F.clean_claims(claims).merge(policies, on="policy_id", how="left")
print(df.shape, "| fraud rate:", round(df["is_fraud"].mean(), 4))
df.head(3)

### Log transformation
Claim amounts are right-skewed: most are small, a few are huge. `log1p(x) = log(1 + x)` pulls the tail in.
Compare the skew before and after (0 = symmetric).

In [ ]:
df["claim_amount_log"] = np.log1p(df["claim_amount"])
df[["claim_amount", "claim_amount_log"]].agg(["mean", "median", "max", "skew"]).round(2)

### Ratio features
A USD 9,000 claim means something different for a USD 10,000 car than for a USD 90,000 car.
Compare the average ratio for fraud and non-fraud claims.

In [ ]:
df["claim_to_value_ratio"] = df["claim_amount"] / df["vehicle_value"]
df["claim_to_premium_ratio"] = df["claim_amount"] / df["annual_premium"]
df.groupby("is_fraud")[["claim_to_value_ratio", "claim_to_premium_ratio"]].mean().round(3)

### Feature splitting
Split one rich column into parts a model can use. A date becomes year / month / weekday;
the year then gives `vehicle_age`. A full name splits into first and last name (shown only as an example).

In [ ]:
date = pd.to_datetime(df["claim_date"])
df["claim_year"], df["claim_month"], df["claim_weekday"] = date.dt.year, date.dt.month, date.dt.dayofweek
df["vehicle_age"] = df["claim_year"] - df["vehicle_year"]
names = df["customer_name"].str.split(" ", n=1, expand=True).rename(columns={0: "first_name", 1: "last_name"})
pd.concat([df[["claim_date", "claim_year", "claim_month", "claim_weekday", "vehicle_age"]], names], axis=1).head(3)

### Binning
- `pd.cut` = bins with fixed edges (age bands). `pd.qcut` = bins with equal numbers of rows (quartiles).
- Look at the fraud rate per bin: it climbs with the amount quartile but is flat across age bands.
  Binning is also a quick way to see whether a feature carries signal.

In [ ]:
df["age_band"] = pd.cut(df["age"], bins=[17, 24, 39, 59, 120], labels=["18-24", "25-39", "40-59", "60+"])
df["amount_quartile"] = pd.qcut(df["claim_amount"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
print(df.groupby("age_band", observed=True)["is_fraud"].mean().round(4).to_dict())
print(df.groupby("amount_quartile", observed=True)["is_fraud"].mean().round(4).to_dict())

### Binary flags (a two-bin binning)
Ideas from the fraud team: young drivers, night-time incidents, brand-new customers and claims without
a police report. The table shows the fraud rate for flag = 0 and 1: three flags separate well, `young_driver`
does not. A feature idea is a hypothesis; the data decides.

In [ ]:
df["young_driver"] = (df["age"] < 25).astype(int)
df["night_incident"] = ((df["incident_hour"] < 5) | (df["incident_hour"] > 22)).astype(int)
df["new_customer"] = (df["tenure_months"] < 6).astype(int)
df["no_police_report"] = (df["police_report"] == "N").astype(int)
flags = ["young_driver", "night_incident", "new_customer", "no_police_report"]
pd.DataFrame({f: df.groupby(f)["is_fraud"].mean() for f in flags}).round(4)

### One-hot encoding
`claim_type`, `channel` and `region` have no order, so each value becomes its own 0/1 column.

In [ ]:
onehot = pd.get_dummies(df[["claim_type", "channel", "region"]], dtype=int)
print(list(onehot.columns))
onehot.head(3)

### Ordinal (label) encoding
`credit_band` A to D **has** an order, so one integer column keeps that information.
(Not used in our model; shown because the exam contrasts it with one-hot encoding.)

In [ ]:
credit_order = {"A": 0, "B": 1, "C": 2, "D": 3}
df["credit_band_ordinal"] = df["credit_band"].map(credit_order)
df[["credit_band", "credit_band_ordinal"]].drop_duplicates().sort_values("credit_band_ordinal")

**📘 Concept: Scaling, standardization, normalization - and data leakage**

The words are used loosely, so learn the formulas:

| Name | Formula (per column) | Result | Use when |
|---|---|---|---|
| **Standardization** (z-score) | (x − mean) / std | mean 0, std 1 | Default for linear models, SVM, PCA, neural networks |
| **Min-max scaling** (often called *normalization*) | (x − min) / (max − min) | range 0 to 1 | A bounded range is needed (pixel values, some neural networks); sensitive to outliers |
| **Robust scaling** | (x − median) / IQR | centred on the median | The column has outliers |
| **Normalization of a row** (L2) | row / length of row | every row has length 1 | Text vectors and embeddings, cosine similarity |

**Who needs it?** Models that use distances or gradients (k-NN, k-means, SVM, linear and logistic regression with
regularization, PCA, neural networks). **Who does not?** Tree models (decision tree, random forest, XGBoost): they only compare a
feature with a threshold.

**Data leakage.** The mean, std, min and max are *learned* from data. If you compute them on the full dataset, information
from the validation and test rows leaks into training and the test score is too optimistic. Correct order:
split → `fit` the scaler on the **training** set → `transform` train, validation, test and later the live requests with
those same values. The same rule applies to imputation values, encoders and feature selection.

Exam cue: "model does well offline but worse in production" or "scaler fitted before the split" → leakage.

### Scaling: standardization, min-max, robust
Fit the scaler on the **training rows only** and reuse it for other data. Compare the resulting ranges.
XGBoost does not need scaling, so the saved features stay unscaled; a linear model or neural network would need it.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

num_cols = ["claim_amount", "age", "tenure_months"]
train_rows = df.sample(frac=0.7, random_state=42)
rows = {}
for name, scaler in [("standard (z-score)", StandardScaler()), ("min-max 0..1", MinMaxScaler()), ("robust (median/IQR)", RobustScaler())]:
    scaled = scaler.fit(train_rows[num_cols]).transform(df[num_cols])
    rows[name] = {"mean": scaled[:, 0].mean(), "std": scaled[:, 0].std(), "min": scaled[:, 0].min(), "max": scaled[:, 0].max()}
pd.DataFrame(rows).T.round(2)       # statistics of the scaled claim_amount column

### Normalization of rows
`Normalizer` rescales each **row** to length 1 (used for text vectors and embeddings before cosine similarity).
Careful with words: many exam questions say "normalization" for min-max scaling of a column.

In [ ]:
from sklearn.preprocessing import Normalizer

unit = Normalizer(norm="l2").fit_transform(df[num_cols].head(3))
print(unit.round(4))
print("row lengths:", np.linalg.norm(unit, axis=1).round(3))

### Put the features together and compare with `common/features.py`
Select the final columns in the agreed order (`F.FEATURES`). The result must equal `F.build_features`.

In [ ]:
mine = pd.concat([df, onehot], axis=1)[F.FEATURES].astype(float).round(5)
feats = F.build_features(claims, policies)
print("same as F.build_features:", np.allclose(mine.values, feats[F.FEATURES].values))
print(len(F.FEATURES), "features:", F.FEATURES[:8], "...")
feats.head(3)

### Split into train / validation / test
70 / 15 / 15, **stratified** on the label so every part has the same fraud rate. `claim_id` is dropped: an ID is not a feature.
This is the same split as `C.ensure_training_data` (same `random_state`).

In [ ]:
from sklearn.model_selection import train_test_split

data = feats.drop(columns="claim_id")
train, rest = train_test_split(data, test_size=0.3, stratify=data[F.LABEL], random_state=42)
val, test = train_test_split(rest, test_size=0.5, stratify=rest[F.LABEL], random_state=42)
pd.DataFrame({name: {"rows": len(part), "fraud_rate": round(part[F.LABEL].mean(), 4)}
              for name, part in [("train", train), ("validation", val), ("test", test)]})

### Write the datasets to S3
SageMaker XGBoost wants CSV with the **label in the first column and no header**. The baseline file
(features only, with header) is for Model Monitor and Clarify later.
If a catch-up run from a later notebook already wrote these files, they are reused.

In [ ]:
REUSE = bool(U.get_state("train_s3"))
if not REUSE:
    U.write_df(feats, f"{PROCESSED}/features.parquet", fmt="parquet")
    uris = {}
    for name, part in [("train", train), ("validation", val), ("test", test)]:
        uris[f"{name}_s3"] = U.write_df(part, f"{PROCESSED}/{name}/{name}.csv", header=False)
    uris["baseline_s3"] = U.write_df(train.drop(columns=F.LABEL), f"{PROCESSED}/baseline/baseline.csv")
    U.save_state(feature_names=F.FEATURES, features_s3=U.s3_uri(f"{PROCESSED}/features.parquet"), **uris)
print("reused existing files" if REUSE else "wrote new files")
{k: v for k, v in U.load_state().items() if k.endswith("_s3")}

Check one file: first column is the label, no header row.

In [ ]:
print(U.read_bytes(f"{PROCESSED}/train/train.csv")[:160].decode())

## 1.2.1 Transform data with AWS tools

The same kind of transformation, now run by managed services. Each job below is small (minutes, a few cents).

| Tool | You write | Runs on | Typical use |
|---|---|---|---|
| AWS Glue job | PySpark / Python script | Serverless Spark (DPUs) | Scheduled ETL, joins of big tables |
| Glue DataBrew | A recipe (list of steps) | Serverless | No-code cleaning, profiling |
| SageMaker Processing | Any script / container | ML instances | Pre-processing inside ML pipelines |
| Amazon EMR | Spark / Hive / Presto code | Your cluster (EC2, EKS, Serverless) | Very large or custom big-data jobs |
| SageMaker Data Wrangler | Visual flow | SageMaker Canvas | Fast exploration and export |

### Glue ETL job: the PySpark script
Reads the two CSV files, joins them, adds three features and writes Parquet partitioned by `claim_type`.

In [ ]:
%%writefile scripts/glue_claims_etl.py
import sys
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql import functions as fn

args = getResolvedOptions(sys.argv, ["JOB_NAME", "claims_path", "policies_path", "output_path"])
glue_context = GlueContext(SparkContext.getOrCreate())
spark = glue_context.spark_session
job = Job(glue_context)
job.init(args["JOB_NAME"], args)

claims = spark.read.csv(args["claims_path"], header=True, inferSchema=True)
policies = spark.read.csv(args["policies_path"], header=True, inferSchema=True)

df = (claims.join(policies, on="policy_id", how="left")
      .withColumn("claim_amount_log", fn.log1p("claim_amount"))
      .withColumn("claim_to_value_ratio", fn.col("claim_amount") / fn.col("vehicle_value"))
      .withColumn("no_police_report", (fn.col("police_report") == "N").cast("int")))

df.write.mode("overwrite").partitionBy("claim_type").parquet(args["output_path"])
print("rows written:", df.count())
job.commit()

### Glue ETL job: upload the script and create the job
A Glue job is a definition: script location, role, worker type and default arguments. 2 × G.1X workers is the smallest Spark job.

In [ ]:
glue = U.client("glue")
GLUE_JOB = "safedrive-claims-etl"
GLUE_OUT = f"{PROCESSED}/glue_claims/"
U.client("s3").upload_file("scripts/glue_claims_etl.py", BUCKET, "safedrive/scripts/glue_claims_etl.py")
if not exists(glue.get_job, JobName=GLUE_JOB):
    glue.create_job(
        Name=GLUE_JOB, Role=ROLE, GlueVersion="5.0", WorkerType="G.1X", NumberOfWorkers=2,
        Timeout=15, MaxRetries=0, Tags=TAG_MAP,
        Command={"Name": "glueetl", "PythonVersion": "3",
                 "ScriptLocation": U.s3_uri("safedrive/scripts/glue_claims_etl.py")},
        DefaultArguments={"--claims_path": U.s3_uri(f"{RAW}/claims.csv"),
                          "--policies_path": U.s3_uri(f"{RAW}/policies.csv"),
                          "--output_path": U.s3_uri(GLUE_OUT)})
print(glue.get_job(JobName=GLUE_JOB)["Job"]["Command"])

### Glue ETL job: start a run

In [ ]:
glue_run = glue.start_job_run(JobName=GLUE_JOB)["JobRunId"]
print(glue_run)

Wait for the run (2 to 4 minutes; most of it is Spark start-up).

In [ ]:
U.wait_until(lambda: glue.get_job_run(JobName=GLUE_JOB, RunId=glue_run)["JobRun"]["JobRunState"],
             ok=("SUCCEEDED",), bad=("FAILED", "ERROR", "TIMEOUT", "STOPPED"), delay=20, label="glue job")

> 🖥️ **Console check:** *AWS Glue → ETL jobs → `safedrive-claims-etl` → Runs tab*.
> Open the run: duration, worker type and number, DPU hours (the cost), and links to the CloudWatch output and error logs.
> The *Script* tab shows the PySpark code, the *Job details* tab the job parameters.

### Glue ETL job: inspect the run and the output
Look at the run time and capacity (billing = DPU-hours), then at the partition folders `claim_type=.../`.

In [ ]:
run = glue.get_job_run(JobName=GLUE_JOB, RunId=glue_run)["JobRun"]
print("seconds:", run.get("ExecutionTime"), "| workers:", run.get("NumberOfWorkers"), run.get("WorkerType"))
out_keys = [k for k in U.list_keys(GLUE_OUT) if k.endswith(".parquet")]
print(sorted({k.split("/")[-2] for k in out_keys}))
U.read_parquet(out_keys[0])[["claim_id", "claim_amount", "claim_amount_log", "claim_to_value_ratio", "region"]].head(3)

### DataBrew: create a dataset
A DataBrew **dataset** is a pointer to data (S3 file, Glue table, JDBC). We point it at the messy `claims_v2.parquet`.

In [ ]:
brew = U.client("databrew")
BREW_DS = "safedrive-claims-v2"
if not exists(brew.describe_dataset, Name=BREW_DS):
    brew.create_dataset(Name=BREW_DS, Format="PARQUET", Tags=TAG_MAP,
                        Input={"S3InputDefinition": {"Bucket": BUCKET, "Key": f"{RAW}/claims_v2.parquet"}})
print(brew.describe_dataset(Name=BREW_DS)["Input"])

### DataBrew: create a recipe
A **recipe** is an ordered list of steps. Each step is an operation name plus string parameters:
remove duplicate rows, trim and lower-case `claim_type`, fill missing amounts with the median,
add a log column, one-hot encode `channel`.

In [ ]:
RECIPE = "safedrive-claims-recipe"
steps = [
    {"Action": {"Operation": "DELETE_DUPLICATE_ROWS"}},
    {"Action": {"Operation": "REMOVE_COMBINED", "Parameters": {
        "sourceColumn": "claim_type", "removeLeadingAndTrailingWhitespace": "true"}}},
    {"Action": {"Operation": "LOWER_CASE", "Parameters": {"sourceColumn": "claim_type"}}},
    {"Action": {"Operation": "FILL_WITH_MEDIAN", "Parameters": {"sourceColumn": "claim_amount"}}},
    {"Action": {"Operation": "LN", "Parameters": {"sourceColumn": "claim_amount", "targetColumn": "claim_amount_ln"}}},
    {"Action": {"Operation": "ONE_HOT_ENCODING", "Parameters": {"sourceColumn": "channel"}}},
]
if not exists(brew.describe_recipe, Name=RECIPE):
    brew.create_recipe(Name=RECIPE, Steps=steps, Tags=TAG_MAP)
print(len(steps), "steps in", RECIPE)

### DataBrew: publish the recipe
A job can only use a **published** recipe version. Publishing gives the working copy a version number.

In [ ]:
brew.publish_recipe(Name=RECIPE, Description="clean and encode claims")
recipe_version = brew.describe_recipe(Name=RECIPE, RecipeVersion="LATEST_PUBLISHED")["RecipeVersion"]
print("published version:", recipe_version)

### DataBrew: create the recipe job
Job = dataset + recipe version + output location. An old job with the same name is replaced.

In [ ]:
BREW_JOB = "safedrive-claims-recipe-job"
BREW_OUT = f"{PROCESSED}/databrew/"
if exists(brew.describe_job, Name=BREW_JOB):
    brew.delete_job(Name=BREW_JOB)
brew.create_recipe_job(
    Name=BREW_JOB, DatasetName=BREW_DS, RoleArn=ROLE, MaxCapacity=2, Timeout=30, Tags=TAG_MAP,
    RecipeReference={"Name": RECIPE, "RecipeVersion": recipe_version},
    Outputs=[{"Location": {"Bucket": BUCKET, "Key": BREW_OUT}, "Format": "CSV", "Overwrite": True}])
print("created", BREW_JOB)

### DataBrew: run the job and wait
About 3 to 5 minutes.

In [ ]:
brew_run = brew.start_job_run(Name=BREW_JOB)["RunId"]
U.wait_until(lambda: brew.describe_job_run(Name=BREW_JOB, RunId=brew_run)["State"],
             ok=("SUCCEEDED",), bad=("FAILED", "STOPPED", "TIMEOUT"), delay=30, label="databrew job")

> 🖥️ **Console check:** *AWS Glue DataBrew → Recipes → `safedrive-claims-recipe`*: the published version and its steps in plain words.
> Then *Jobs → `safedrive-claims-recipe-job`* for the run history and output location. This no-code view is what an analyst would work in.

### DataBrew: inspect the output
Compare with the raw file: fewer rows (duplicates gone), no missing amounts, clean `claim_type`, new columns.

In [ ]:
raw_v2 = U.read_parquet(f"{RAW}/claims_v2.parquet")
brew_csv = [k for k in U.list_keys(BREW_OUT) if k.endswith(".csv")]
cleaned = pd.concat([U.read_csv(k) for k in brew_csv])
print("rows:", len(raw_v2), "->", len(cleaned), "| missing amounts:", raw_v2["claim_amount"].isna().sum(), "->", cleaned["claim_amount"].isna().sum())
print(sorted(cleaned["claim_type"].unique()))
print([c for c in cleaned.columns if c not in raw_v2.columns])

### SageMaker Processing: the script
A Processing job copies inputs from S3 to `/opt/ml/processing/...`, runs your script in a container and
uploads whatever the script writes to the output folders. The script imports our `features.py`, so the
job and the notebook share one feature definition.

In [ ]:
%%writefile scripts/preprocess.py
import argparse
import os
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

BASE = "/opt/ml/processing"
sys.path.append(f"{BASE}/lib")
import features as F  # noqa: E402  (common/features.py, sent as an input)

parser = argparse.ArgumentParser()
parser.add_argument("--test-size", type=float, default=0.2)
args = parser.parse_args()

claims = pd.read_csv(f"{BASE}/input/claims/claims.csv")
policies = pd.read_csv(f"{BASE}/input/policies/policies.csv")
feats = F.build_features(claims, policies).drop(columns="claim_id")
train, test = train_test_split(feats, test_size=args.test_size, stratify=feats[F.LABEL], random_state=42)

for name, part in [("train", train), ("test", test)]:
    os.makedirs(f"{BASE}/{name}", exist_ok=True)
    part.to_csv(f"{BASE}/{name}/{name}.csv", index=False, header=False)
    print(name, part.shape, "fraud rate", round(part[F.LABEL].mean(), 4))

### SageMaker Processing: define the processor
`SKLearnProcessor` uses the AWS scikit-learn container. One `ml.m5.large` is plenty.

In [ ]:
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor

sm_session = sagemaker.Session(boto_session=U.session, default_bucket=BUCKET)
processor = SKLearnProcessor(framework_version="1.2-1", role=ROLE, instance_type="ml.m5.large",
                             instance_count=1, base_job_name="safedrive-preprocess",
                             max_runtime_in_seconds=900, sagemaker_session=sm_session, tags=U.TAGS)
print(processor.image_uri)

### SageMaker Processing: inputs and outputs
Each input is an S3 location (or local file) mapped to a container folder; each output is the reverse.

In [ ]:
IN, PROC_OUT = "/opt/ml/processing", f"{PROCESSED}/processing-demo"
inputs = [
    ProcessingInput(source=U.s3_uri(f"{RAW}/claims.csv"), destination=f"{IN}/input/claims"),
    ProcessingInput(source=U.s3_uri(f"{RAW}/policies.csv"), destination=f"{IN}/input/policies"),
    ProcessingInput(source=os.path.join(U.REPO_ROOT, "common", "features.py"), destination=f"{IN}/lib"),
]
outputs = [
    ProcessingOutput(source=f"{IN}/train", destination=U.s3_uri(f"{PROC_OUT}/train"), output_name="train"),
    ProcessingOutput(source=f"{IN}/test", destination=U.s3_uri(f"{PROC_OUT}/test"), output_name="test"),
]
print(len(inputs), "inputs,", len(outputs), "outputs")

### SageMaker Processing: run the job
About 4 to 5 minutes (instance start + container download). `wait=True` blocks until it ends.

In [ ]:
processor.run(code="scripts/preprocess.py", inputs=inputs, outputs=outputs,
              arguments=["--test-size", "0.2"], wait=True, logs=False)
print(processor.latest_job.describe()["ProcessingJobStatus"])

> 🖥️ **Console check:** *Amazon SageMaker AI → Processing → Processing jobs → `safedrive-preprocess-...`*.
> Look at the instance type, the input and output S3 mappings, the run time, and the *View logs* link to CloudWatch (the `print` lines of the script).

### SageMaker Processing: inspect the output
These demo files go to `processing-demo/`; the model in Domain 2 trains on the files written in section 1.2.4.

In [ ]:
for k in U.list_keys(PROC_OUT + "/"):
    print(k)
U.read_csv(f"{PROC_OUT}/train/train.csv", header=None).iloc[:3, :6]

### EMR and Data Wrangler (notes)

| Tool | Remember for the exam |
|---|---|
| **Amazon EMR** | Managed Spark / Hive / Presto / Flink. Options: EMR on EC2 (most control, Spot for task nodes), EMR on EKS, **EMR Serverless** (no cluster to manage). Use it for petabyte-scale or existing Spark code. SageMaker Studio notebooks can connect to an EMR cluster. |
| **SageMaker Data Wrangler** | Visual flow in SageMaker Canvas: import (S3, Athena, Redshift, Snowflake), 300+ transforms, **data quality and insights report**, quick model, bias report. Export a flow to a Processing job, a Pipeline step, Feature Store or a notebook. |
| **Glue interactive sessions / Studio notebooks** | Develop Spark code interactively before turning it into a job. |

```python
# EMR Serverless: submit a Spark job (shape of the call)
emr = boto3.client("emr-serverless")
emr.start_job_run(applicationId=app_id, executionRoleArn=role,
    jobDriver={"sparkSubmit": {"entryPoint": "s3://bucket/scripts/etl.py",
                               "entryPointArguments": ["--output", "s3://bucket/out/"]}})
```

## 1.2.2 Create and manage features with SageMaker Feature Store

- **Online store**: latest value per record ID, single-digit-millisecond `GetRecord` / `BatchGetRecord`.
- **Offline store**: append-only history in S3 (Parquet / Iceberg) + Glue table → query with Athena, build training sets.
- **Point-in-time (time travel) query**: for each training row use only feature values known *at that time* → no label leakage.
- Manage: feature descriptions and parameters (searchable), add features with `UpdateFeatureGroup`, TTL for online records, delete records.

### Create the feature group if notebook 1.1 was skipped
Same definition as in 1.1: record ID `claim_id`, event time `event_time`, online and offline stores.

In [ ]:
sm = U.client("sagemaker")
FG = "safedrive-claims-fg"
feats_fs = F.build_features(claims, policies)
feats_fs["event_time"] = feats_fs["claim_id"].map(claims.set_index("claim_id")["claim_date"]) + "T00:00:00Z"
NEW_FG = not exists(sm.describe_feature_group, FeatureGroupName=FG)
if NEW_FG:
    types = {"claim_id": "String", "event_time": "String", F.LABEL: "Integral"}
    sm.create_feature_group(
        FeatureGroupName=FG, RecordIdentifierFeatureName="claim_id", EventTimeFeatureName="event_time",
        FeatureDefinitions=[{"FeatureName": c, "FeatureType": types.get(c, "Fractional")} for c in feats_fs.columns],
        RoleArn=ROLE, Tags=U.TAGS, OnlineStoreConfig={"EnableOnlineStore": True},
        OfflineStoreConfig={"S3StorageConfig": {"S3Uri": U.s3_uri("safedrive/feature-store")}})
U.wait_until(lambda: sm.describe_feature_group(FeatureGroupName=FG)["FeatureGroupStatus"],
             ok=("Created",), bad=("CreateFailed",), delay=10, label="feature group")

Ingest 2,000 records if the group was just created (otherwise notebook 1.1 already did it).

In [ ]:
from sagemaker.feature_store.feature_group import FeatureGroup

if NEW_FG:
    FeatureGroup(name=FG, sagemaker_session=sm_session).ingest(feats_fs.head(2000), max_workers=4, wait=True)
U.save_state(feature_group_name=FG)
print("newly created and ingested" if NEW_FG else "reused existing feature group")

### Describe the feature group
Note the two stores and the Glue table that the offline store registered.

In [ ]:
desc = sm.describe_feature_group(FeatureGroupName=FG)
catalog = desc["OfflineStoreConfig"]["DataCatalogConfig"]
print("status:", desc["FeatureGroupStatus"], "| features:", len(desc["FeatureDefinitions"]))
print("online:", desc["OnlineStoreConfig"])
print("offline:", desc["OfflineStoreConfig"]["S3StorageConfig"]["ResolvedOutputS3Uri"])
print("glue table:", catalog["Database"], catalog["TableName"])

### Online store: get one record
This is the call an inference Lambda makes just before invoking the model.

In [ ]:
fsr = U.client("sagemaker-featurestore-runtime")
CLAIM = feats_fs["claim_id"].iloc[5]
rec = fsr.get_record(FeatureGroupName=FG, RecordIdentifierValueAsString=CLAIM,
                     FeatureNames=["claim_id", "claim_amount_log", "witnesses", "event_time"])
{f["FeatureName"]: f["ValueAsString"] for f in rec["Record"]}

### Online store: get many records in one call
`BatchGetRecord` can read several IDs (and several feature groups) at once.

In [ ]:
ids = feats_fs["claim_id"].iloc[:3].tolist()
batch = fsr.batch_get_record(Identifiers=[{
    "FeatureGroupName": FG, "RecordIdentifiersValueAsString": ids,
    "FeatureNames": ["claim_id", "claim_to_value_ratio", "prior_claims"]}])
pd.DataFrame([{f["FeatureName"]: f["ValueAsString"] for f in r["Record"]} for r in batch["Records"]])

### Update a record
Features change over time. Write the same `claim_id` again with a **newer event time** and one more witness.
The online store now returns the new version; the offline store keeps both.

In [ ]:
row = feats_fs[feats_fs["claim_id"] == CLAIM].iloc[0].copy()
row["witnesses"] = row["witnesses"] + 1
row["event_time"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
fsr.put_record(FeatureGroupName=FG,
               Record=[{"FeatureName": c, "ValueAsString": str(row[c])} for c in feats_fs.columns])
rec = fsr.get_record(FeatureGroupName=FG, RecordIdentifierValueAsString=CLAIM, FeatureNames=["witnesses", "event_time"])
{f["FeatureName"]: f["ValueAsString"] for f in rec["Record"]}

### Manage feature metadata
Descriptions and key-value parameters make features searchable in SageMaker Studio and document who owns them.

In [ ]:
sm.update_feature_metadata(
    FeatureGroupName=FG, FeatureName="claim_to_value_ratio",
    Description="claim_amount divided by insured vehicle value",
    ParameterAdditions=[{"Key": "owner", "Value": "fraud-team"}, {"Key": "pii", "Value": "no"}])
time.sleep(3)
meta = sm.describe_feature_metadata(FeatureGroupName=FG, FeatureName="claim_to_value_ratio")
print(meta["FeatureType"], "|", meta.get("Description"), "|", meta.get("Parameters"))

> 🖥️ **Console check:** In **SageMaker Studio**: *Data → Feature Store → `safedrive-claims-fg`*.
> Open the feature `claim_to_value_ratio`: the description and the `owner` / `pii` parameters you just set are shown and are searchable.

### Offline store: a small Athena helper
Start the query, wait, then read the result file Athena wrote to S3.

In [ ]:
ath = U.client("athena")
ATHENA_OUT = "safedrive/athena/"

def athena_df(sql, database):
    qid = ath.start_query_execution(
        QueryString=sql, QueryExecutionContext={"Database": database},
        ResultConfiguration={"OutputLocation": U.s3_uri(ATHENA_OUT)})["QueryExecutionId"]
    U.wait_until(lambda: ath.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"],
                 ok=("SUCCEEDED",), bad=("FAILED", "CANCELLED"), delay=2, label="athena")
    return U.read_csv(f"{ATHENA_OUT}{qid}.csv")

FS_DB, FS_TABLE = catalog["Database"], catalog["TableName"]
athena_df(f'SELECT count(*) AS records, count(DISTINCT claim_id) AS claims FROM "{FS_TABLE}"', FS_DB)

> Records reach the offline store a few minutes after ingestion (up to about 15). If the count is 0, wait and run the cell again.

### Offline store: history of one record
Feature Store added three columns: `write_time`, `api_invocation_time`, `is_deleted`.
Our updated claim will show two rows once the second write has landed.

In [ ]:
sql = f"""
SELECT claim_id, witnesses, event_time, write_time, is_deleted
FROM "{FS_TABLE}" WHERE claim_id = '{CLAIM}' ORDER BY event_time
"""
athena_df(sql, FS_DB)

**📘 Concept: Online store, offline store and point-in-time correctness**

| | Online store | Offline store |
|---|---|---|
| Holds | Only the **latest** record per ID | **Every** record ever written (append-only history) |
| Read with | `GetRecord` / `BatchGetRecord`, milliseconds | Athena / Spark on Parquet files in S3, seconds to minutes |
| Used for | Real-time inference | Building training sets, batch scoring, analysis |
| Freshness | Immediately after `PutRecord` | A few minutes later |

Both are filled by the same `PutRecord`, so training and serving see features computed by **one** piece of code. That removes
*training-serving skew*.

**Point-in-time correctness.** A training row describes an event in the past (a claim filed on 10 March). Its features must be the
values that were known **on that day**, not today's values. Example: `prior_claims` was 1 on 10 March and is 3 today. Training on 3
tells the model something it could not have known when predicting: **label leakage** from the future. A point-in-time (as-of) query takes,
for each row, the newest feature record whose **event time** is not later than the row's own timestamp.

Exam cue: "features as they were at the time of the event", "time travel", "avoid leakage when joining feature history" →
offline store with a point-in-time join on event time.

### Offline store: point-in-time query
"What did we know about each claim on 30 June 2025?" Keep rows with `event_time` up to the cutoff, then
take the newest row per `claim_id`. Later updates (like the one we just wrote) are ignored: no leakage from the future.

In [ ]:
CUTOFF = "2025-06-30T23:59:59Z"
sql = f"""
SELECT claim_id, witnesses, claim_amount_log, is_fraud, event_time FROM (
  SELECT *, row_number() OVER (PARTITION BY claim_id ORDER BY event_time DESC, write_time DESC) AS rn
  FROM "{FS_TABLE}" WHERE event_time <= '{CUTOFF}' AND NOT is_deleted)
WHERE rn = 1 ORDER BY event_time DESC LIMIT 5
"""
athena_df(sql, FS_DB)

### More Feature Store management (notes)

| Task | How |
|---|---|
| Add a feature | `update_feature_group(FeatureAdditions=[...])` (features cannot be removed or retyped) |
| Build a training set from several groups | SDK `FeatureStore.create_dataset(...).point_in_time_accurate_join().to_csv_file()` |
| Expire old online records | `TtlDuration` on the group or per `PutRecord` |
| Delete a record | `delete_record` (soft delete; `HardDelete` removes it from the online store) |
| Big backfills | Feature Store Spark connector or `ingest()` with more workers; target only the offline store with `TargetStores=["OfflineStore"]` |
| Fresher / cheaper online tier | `StorageType`: `Standard` or `InMemory` |
| Compact offline data | Iceberg table format |

## 1.2.3 Transform streaming data

- **Lambda** fits simple, stateless, per-record or per-batch work: parse, filter, enrich, mask.
  Two ways to attach it: a **Firehose data-transformation** function, or a **Kinesis event source mapping**.
- **Spark Structured Streaming** (Glue streaming jobs, EMR) and **Managed Service for Apache Flink** fit
  stateful work: time windows, aggregations, stream joins.

We use Firehose (Direct PUT) → Lambda → S3. The function adds risk flags to each telematics event,
removes the precise location, and drops broken records.

### The Lambda function
Firehose sends a batch of base64 records. For each `recordId` the function must answer with
`Ok` (transformed data), `Dropped` or `ProcessingFailed`.

In [ ]:
%%writefile scripts/telematics_transform.py
import base64
import json


def transform(e):
    e["speeding"] = int(e["speed_kmh"] > 120)
    e["risk_event"] = int(e["speeding"] == 1 or e["harsh_brake"] == 1 or abs(e["accel_g"]) > 0.5)
    e["hour"] = int(e["ts"][11:13])
    e.pop("lat", None)      # drop the precise location (privacy)
    e.pop("lon", None)
    return e


def handler(event, context):
    out = []
    for rec in event["records"]:
        try:
            e = transform(json.loads(base64.b64decode(rec["data"])))
            data = base64.b64encode((json.dumps(e) + "\n").encode()).decode()
            out.append({"recordId": rec["recordId"], "result": "Ok", "data": data})
        except (KeyError, TypeError, ValueError):
            out.append({"recordId": rec["recordId"], "result": "Dropped", "data": rec["data"]})
    return {"records": out}

### Deploy the function
Zip the file in memory and create the function (or update the code if it exists).

In [ ]:
import zipfile

lam = U.client("lambda")
FN = "safedrive-telematics-transform"
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as z:
    z.write("scripts/telematics_transform.py", "telematics_transform.py")
if exists(lam.get_function, FunctionName=FN):
    lam.update_function_code(FunctionName=FN, ZipFile=buf.getvalue())
else:
    lam.create_function(FunctionName=FN, Runtime="python3.12", Role=ROLE, Timeout=60, Tags=TAG_MAP,
                        Handler="telematics_transform.handler", Code={"ZipFile": buf.getvalue()})
lam.get_waiter("function_active_v2").wait(FunctionName=FN)
FN_ARN = lam.get_function(FunctionName=FN)["Configuration"]["FunctionArn"]
print(FN_ARN)

### Test the function directly
Build a fake Firehose event from two real telematics events plus one broken record, and invoke the function.
Expect two `Ok` results (with the new fields, without `lat` / `lon`) and one `Dropped`.

In [ ]:
import base64

events = U.read_jsonl(f"{RAW}/telematics_stream.jsonl")
payloads = [json.dumps(events[0]), json.dumps(events[1]), "not json"]
test_event = {"records": [{"recordId": str(i), "data": base64.b64encode(p.encode()).decode()}
                          for i, p in enumerate(payloads)]}
lam.get_waiter("function_updated_v2").wait(FunctionName=FN)
result = json.loads(lam.invoke(FunctionName=FN, Payload=json.dumps(test_event))["Payload"].read())
for r in result["records"]:
    print(r["recordId"], r["result"], base64.b64decode(r["data"]).decode().strip()[:150])

### Create a Firehose stream that calls the function
`ProcessingConfiguration` attaches the Lambda function. Producers write straight to Firehose (`DirectPut`).

In [ ]:
fh = U.client("firehose")
FIREHOSE = "safedrive-telematics-transform"
OUT_PREFIX = "safedrive/streams/telematics-transformed/"
if not exists(fh.describe_delivery_stream, DeliveryStreamName=FIREHOSE):
    fh.create_delivery_stream(
        DeliveryStreamName=FIREHOSE, DeliveryStreamType="DirectPut", Tags=U.TAGS,
        ExtendedS3DestinationConfiguration={
            "RoleARN": ROLE, "BucketARN": f"arn:aws:s3:::{BUCKET}", "Prefix": OUT_PREFIX,
            "ErrorOutputPrefix": "safedrive/streams/errors/",
            "BufferingHints": {"SizeInMBs": 1, "IntervalInSeconds": 60},
            "ProcessingConfiguration": {"Enabled": True, "Processors": [{
                "Type": "Lambda",
                "Parameters": [{"ParameterName": "LambdaArn", "ParameterValue": f"{FN_ARN}:$LATEST"}]}]}})
print("requested", FIREHOSE)

Wait until the Firehose stream is `ACTIVE`.

In [ ]:
U.wait_until(
    lambda: fh.describe_delivery_stream(DeliveryStreamName=FIREHOSE)["DeliveryStreamDescription"]["DeliveryStreamStatus"],
    ok=("ACTIVE",), bad=("CREATING_FAILED",), delay=10, label="firehose")

### Send events
`put_record_batch` takes up to 500 records. Check `FailedPutCount` (throttled records must be re-sent).

In [ ]:
before = set(U.list_keys(OUT_PREFIX))
records = [{"Data": json.dumps(e).encode()} for e in events[:500]]
resp = fh.put_record_batch(DeliveryStreamName=FIREHOSE, Records=records)
print("sent:", len(records), "| failed:", resp["FailedPutCount"])

### Wait for the transformed output in S3
Firehose buffers the input (for Lambda) and the output (for S3), so allow 1 to 3 minutes.

In [ ]:
U.wait_until(lambda: len(set(U.list_keys(OUT_PREFIX)) - before) > 0, ok=(True,), delay=20, timeout=420,
             label="new objects delivered")
new_keys = sorted(set(U.list_keys(OUT_PREFIX)) - before)
new_keys

> 🖥️ **Console check:** *Lambda → Functions → `safedrive-telematics-transform` → Monitor tab*: invocations, duration and errors
> (Firehose called the function once per batch, not once per record). Then *Amazon Data Firehose → Firehose streams →
> `safedrive-telematics-transform`*: the *Configuration* tab shows the attached transformation function and the buffer settings.

### Inspect the transformed events
New columns `speeding`, `risk_event`, `hour`; no `lat` / `lon`.

In [ ]:
out = pd.read_json(io.BytesIO(U.read_bytes(new_keys[0])), lines=True)
print(out.shape, "| risk events:", int(out["risk_event"].sum()))
out.head(3)

### Other ways to transform a stream (notes)

| Option | Remember |
|---|---|
| Lambda + **Kinesis event source mapping** | Lambda polls the shards and gets batches (`BatchSize`, `MaximumBatchingWindowInSeconds`, `ParallelizationFactor`); you write the output yourself; failed batches block the shard unless you set retries / bisect / on-failure destination |
| **Glue streaming job** (Spark Structured Streaming) | Micro-batches from Kinesis or Kafka, windows and joins, writes to S3 / catalog tables |
| **EMR** Spark Streaming | Same engine, your cluster |
| **Managed Service for Apache Flink** | True streaming, event-time windows, exactly-once state |

```python
# Lambda on a Kinesis stream (shape of the call)
lam.create_event_source_mapping(EventSourceArn=stream_arn, FunctionName=FN,
                                StartingPosition="LATEST", BatchSize=100,
                                MaximumBatchingWindowInSeconds=5)

# Spark Structured Streaming: harsh brakes per policy in 1-minute windows
events = (spark.readStream.format("kinesis").option("streamName", "safedrive-telematics")
          .option("startingPosition", "LATEST").load())
parsed = events.select(from_json(col("data").cast("string"), schema).alias("e")).select("e.*")
agg = (parsed.withWatermark("ts", "2 minutes")
       .groupBy(window("ts", "1 minute"), "policy_id").agg(sum("harsh_brake").alias("harsh_brakes")))
agg.writeStream.format("parquet").option("path", "s3://bucket/agg/").option("checkpointLocation", "s3://bucket/chk/").start()
```

## Exam traps

- **Scaling is fitted on training data only.** Fitting on the full dataset leaks test information.
- **Tree-based models do not need scaling or one-hot for ordinal data**; distance- and gradient-based models do need scaling.
- **DataBrew** = no-code cleaning for analysts; **Data Wrangler** = ML-focused prep inside SageMaker; **Glue** = code-based Spark ETL.
- A Firehose transformation Lambda must return **every `recordId`** with `Ok`, `Dropped` or `ProcessingFailed`, and the data base64 encoded.
- Feature Store **offline** data is delayed by minutes and append-only; use event time for point-in-time correct training sets.

## Quiz

**1.** A data analyst with no coding skills must remove duplicates, fix date formats and fill missing values in a
50 GB CSV dataset in S3 every week. Which service needs the least effort?
- A. Amazon EMR with a PySpark script
- B. AWS Glue DataBrew recipe job on a schedule
- C. SageMaker Processing with scikit-learn
- D. AWS Lambda triggered by S3

<details><summary>Answer</summary>B. DataBrew recipes are no-code and can be scheduled as jobs. The other options need code (and Lambda cannot handle 50 GB).</details>

**2.** A feature `annual_income` is strongly right-skewed and hurts a linear regression model. Which transformation helps most?
- A. One-hot encoding
- B. Log transformation
- C. Min-max scaling
- D. Binning into 2 bins

<details><summary>Answer</summary>B. A log transform reduces skew. Min-max scaling changes the range but keeps the skewed shape.</details>

**3.** A k-means model performs badly because `vehicle_value` (thousands) dominates `age` (tens). The data has
a few extreme outliers. Which pre-processing is the best choice?
- A. Robust scaling (median and IQR) fitted on the training set
- B. Label encoding
- C. Min-max scaling fitted on the full dataset
- D. No scaling; k-means is scale invariant

<details><summary>Answer</summary>A. Distance-based models need scaling; robust scaling is not distorted by outliers; fit on training data only.</details>

**4.** Telematics events arrive in Amazon Data Firehose and must have the GPS coordinates removed before they
land in S3, with the least operational overhead. What should the ML engineer do?
- A. Run a nightly Glue job that rewrites the S3 objects
- B. Enable Firehose data transformation with an AWS Lambda function
- C. Replace Firehose with an EMR Spark Streaming cluster
- D. Use S3 Object Lambda on every read

<details><summary>Answer</summary>B. A Firehose transformation Lambda changes each record in flight, fully managed.</details>

**5.** A training dataset must use, for every past transaction, the customer feature values as they were at the
time of the transaction. Which approach gives this?
- A. Read the latest values from the Feature Store online store
- B. A point-in-time (as-of event time) query on the Feature Store offline store
- C. Export the DynamoDB table used by the application
- D. Recompute the features with today's data

<details><summary>Answer</summary>B. The offline store keeps history with event times; a point-in-time join avoids leakage from the future.</details>

## Save state
The training-data keys were saved in section 1.2.4; this cell shows what later notebooks will read.

In [ ]:
state = U.load_state()
{k: state.get(k) for k in ("train_s3", "validation_s3", "test_s3", "baseline_s3", "features_s3", "feature_group_name")}

## Cleanup

**Kept for later notebooks** (no idle cost): the training files under `safedrive/processed/`, the feature group
`safedrive-claims-fg`. Everything else created here is deleted below (none of it bills while idle, this is just tidy).

### Before you delete: what later notebooks use

The cleanup cells below **do not delete** the items in this table.

| Kept | Used by later notebooks | If you delete it anyway |
|---|---|---|
| Training files under `safedrive/processed/` (train, validation, test, baseline) | 1.3, 2.2a, 2.3a, 3.1a, 3.2a, 3.3a, 4.1a, 4.2, 4.3a | Rebuilt by the catch-up cell (about a minute) |
| Feature group `safedrive-claims-fg` | Not used again | - |

- **Continuing to the next notebook soon?** Run the cleanup cells and keep the items above. None of them bills while idle (only a little storage).
- **Stopping for good?** Run `99_cleanup/99_teardown.ipynb`; it removes everything, including these.

Delete the Firehose stream.

In [ ]:
U.safe(fh.delete_delivery_stream, DeliveryStreamName=FIREHOSE)

Delete the Lambda function.

In [ ]:
U.safe(lam.delete_function, FunctionName=FN)

Delete the Glue job.

In [ ]:
U.safe(glue.delete_job, JobName=GLUE_JOB)

Delete the DataBrew job.

In [ ]:
U.safe(brew.delete_job, Name=BREW_JOB)

Delete the DataBrew recipe (all published versions plus the working copy).

In [ ]:
versions = [r["RecipeVersion"] for r in brew.list_recipe_versions(Name=RECIPE)["Recipes"]]
U.safe(brew.batch_delete_recipe_version, Name=RECIPE, RecipeVersions=versions + ["LATEST_WORKING"])

Delete the DataBrew dataset (notebook 1.3 creates it again if needed).

In [ ]:
U.safe(brew.delete_dataset, Name=BREW_DS)